In [ ]:
%%bash
# ASL Citizen -> MediaPipe Holistic landmarks for the 250-sign edge-vision vocabulary.
# Settings: Accelerator None (CPU), Internet On. Run with Save Version > Save & Run All (Commit).
# Licence: non-commercial research only, never redistribute, keep this notebook and its outputs
# private, delete everything when the research is complete (docs/DATA_LICENSES.md).
# Generated by scripts/build_kaggle_notebooks.py from the repository sources; edit those.
# Resumable: add an earlier version's output as an input and run again; finished shards are
# skipped. Parallel parts: three copies of this notebook with EV_PART 0/3, 1/3 and 2/3, then one
# run with 0/1 and the three outputs added as inputs, which merges them and, with the
# KAGGLE_USERNAME and KAGGLE_KEY secrets attached, creates the private dataset.
set -u
export EV_LIMIT_VIDEOS=0    # SMOKE TEST switch: > 0 tracks only the first N videos, into a separate -smoke folder
export EV_PART="0/1"         # k/N: this copy's share of the shards (see above)
WORK="${EV_WORK:-/kaggle/working}"    # EV_WORK and EV_INPUT exist for local tests only
export EV_INPUT="${EV_INPUT:-/kaggle/input}"
if [ "$EV_LIMIT_VIDEOS" -gt 0 ]; then
  NAME=asl_citizen_smoke; OUT="$WORK/asl-citizen-landmarks-smoke"
else
  NAME=asl_citizen_extract; OUT="$WORK/asl-citizen-landmarks"
fi
export EV_OUT="$OUT"
LOG="$WORK/$NAME.log"
MPENV=/tmp/mpenv
PY="$MPENV/bin/python"
mkdir -p "$WORK"

# Kaggle's Python is 3.13 and mediapipe 0.10.21 has wheels only up to 3.12, so the extraction runs
# in a Python 3.11 venv made with uv. clean runs that venv's Python without Kaggle's
# site hooks: Kaggle's PYTHONPATH holds a sitecustomize that imports wrapt, which the venv lacks.
clean() { env -u PYTHONPATH -u PYTHONSTARTUP -u PYTHONHOME PYTHONNOUSERSITE=1 "$@"; }
uv() { python -m uv "$@"; }
venv_versions() {
  clean "$PY" -c 'import platform, mediapipe, cv2, numpy; print(f"venv {platform.python_version()}: mediapipe {mediapipe.__version__}, cv2 {cv2.__version__}, numpy {numpy.__version__}")'
}
setup_environment() {
  echo "=== $(date -u '+%F %T') UTC: start $NAME; Kaggle python $(python -c 'import platform; print(platform.python_version())' 2>&1)"
  if [ -x "$PY" ] && venv_versions 2>/dev/null; then
    echo "reusing $MPENV"
    return 0
  fi
  echo "--- pip install uv"
  python -m uv --version 2>/dev/null || python -m pip install --disable-pip-version-check uv \
    || { echo "STEP FAILED: pip install uv"; return 1; }
  echo "--- uv python install 3.11"
  uv python install 3.11 || { echo "STEP FAILED: uv python install 3.11"; return 1; }
  echo "--- uv venv --python 3.11 $MPENV"
  uv venv --clear --python 3.11 "$MPENV" || { echo "STEP FAILED: uv venv"; return 1; }
  echo '--- uv pip install mediapipe==0.10.21 opencv-python-headless "numpy<2" pandas pyarrow'
  uv pip install --python "$PY" mediapipe==0.10.21 opencv-python-headless "numpy<2" pandas pyarrow \
    || { echo "STEP FAILED: uv pip install"; return 1; }
  # mediapipe also requires opencv-contrib-python, which writes the same cv2 package as the
  # headless build. Keep exactly one cv2: drop contrib, then restore the headless files it shared.
  if uv pip show --python "$PY" opencv-contrib-python >/dev/null 2>&1; then
    OPENCV=$(uv pip show --python "$PY" opencv-python-headless | sed -n 's/^Version: //p')
    echo "--- keeping only opencv-python-headless==$OPENCV"
    uv pip uninstall --python "$PY" opencv-contrib-python \
      && uv pip install --python "$PY" --reinstall "opencv-python-headless==$OPENCV" "numpy<2" \
      || { echo "STEP FAILED: replacing opencv-contrib-python with the headless build"; return 1; }
  fi
  echo "--- versions inside $MPENV"
  venv_versions || { echo "STEP FAILED: importing mediapipe, cv2 and numpy in $MPENV"; return 1; }
}
setup_environment 2>&1 | tee -a "$LOG"
if [ "${PIPESTATUS[0]}" -ne 0 ]; then
  echo "ENVIRONMENT FAILED: nothing was extracted. The full error is above and in $LOG." | tee -a "$LOG"
  echo "=== $(date -u '+%F %T') UTC: finished; this cell always exits 0" | tee -a "$LOG"
  exit 0
fi
cat > "$WORK/$NAME.py" <<'PYEOF'
# Generated by scripts/build_kaggle_notebooks.py; edit the sources, not this cell.
# ---- src/edgevision/data/citizen_labels.py ----
"""Map ASL Citizen glosses onto the 250-sign asl-signs vocabulary.

Pure Python without third-party imports: scripts/build_kaggle_notebooks.py inlines this file into
the Kaggle extraction notebook, so the notebook and the repository apply the same mapping.

A gloss maps to a label by the first rule that applies:

1. ``exact``: the gloss and the label are equal once lower-cased with everything except letters
   and digits removed, so THANKYOU, "thank you" and Thank-You all match the label "thankyou".
2. ``variant``: the gloss is the label plus a numeric suffix (CAT1, CAT2 and CAT3 for "cat").
   ASL Citizen numbers the ASL-LEX 2.0 variants of a word. Every variant is mapped unless
   VARIANT_EXCLUDED lists it, which it does where ASL-LEX gives that variant another meaning.
3. ``synonym``: SYNONYMS lists other glosses per label, chosen where ASL-LEX 2.0's English
   translations of the sign name the label's word (mom: MOTHER).

A gloss maps to at most one label (build_mapping raises on a conflict), and the labels in
UNRESOLVED stay unmapped until someone decides; their candidate glosses are in REVIEW_GLOSSES,
which the extraction notebook processes too, so a later decision needs no second extraction.

REVIEW_STATUS applies to VARIANT_EXCLUDED, SYNONYMS and UNRESOLVED: they were chosen with AI
assistance from ASL-LEX 2.0's translations and have not been confirmed by a signer.
"""

import re
from collections.abc import Iterable, Sequence
from dataclasses import dataclass, field

REVIEW_STATUS = "ai-assisted, unconfirmed"
RULES = ("exact", "variant", "synonym")

# Variants whose ASL-LEX 2.0 entry gives them a meaning other than the label's.
VARIANT_EXCLUDED: dict[str, str] = {
    "DRINK2": "ASL-LEX translations: alcohol, beverage, drink (the alcohol sign)",
    "EMPTY2": "ASL-LEX dominant translation: dump",
    "HEAR2": "ASL-LEX dominant translation: listen, which is a label of its own",
    "TALK2": "ASL-LEX dominant translation: duck (the blah-blah gesture); duck is a label",
    "TOY2": "ASL-LEX dominant translation: shave",
}

# label -> (glosses, basis). The basis quotes the ASL-LEX 2.0 English translations.
SYNONYMS: dict[str, tuple[tuple[str, ...], str]] = {
    "alligator": (("CROCODILE",), "translations: alligator, animal, crocodile"),
    "another": (("OTHER",), "translations: another, else, other, otherwise"),
    "beside": (("NEXTTO",), "translations: besides, by, near, next to"),
    "callonphone": (("PHONE", "CALL1"), "translations: call, phone, telephone (both)"),
    "dad": (("FATHER",), "translations: dad, daddy, father, papa"),
    "every": (("EACH",), "translations: apiece, each, every"),
    "eye": (("EYES",), "translations: body part, eyes"),
    "food": (("EAT1", "EAT2"), "translations: consume, food, eat (both)"),
    "garbage": (("TRASH",), "translations: bin, garbage, trash, trash can"),
    "grandma": (("GRANDMOTHER",), "translations: grandma, grandmother, nana"),
    "grandpa": (("GRANDFATHER",), "translations: baba, grandfather, grandpa"),
    "haveto": (("MUST",), "translations: have to, imperative, must, need, should"),
    "hesheit": (("HE",), "dominant translation: he (the pointing pronoun)"),
    "into": (("IN",), "ASL-LEX entry 'in' (no translations listed)"),
    "kitty": (("KITTEN",), "translations: baby cat, kitten, kitty, kitty cat"),
    "look": (("LOOKAT",), "translations: look, look at, see, to watch"),
    "minemy": (("MY",), "ASL-LEX entry 'my' (MINE is the digging sign)"),
    "mitten": (("GLOVE", "GLOVES1", "GLOVES2"), "translations: glove, gloves, mittens"),
    "mom": (("MOTHER",), "translations: mama, mom, mommy, mother"),
    "noisy": (("NOISE",), "translations: noise, sound (LOUD is the label loud)"),
    "nuts": (("NUT1", "NUT2"), "translations: almond, cashew, nut, nuts, peanut"),
    "owie": (("HURT", "PAIN"), "translations: ache, hurt, injury, pain (both)"),
    "police": (("POLICEMAN1", "POLICEMAN2"), "translations: cop, police, police officer"),
    "potty": (("TOILET", "BATHROOM"), "translations: bathroom, restroom, toilet (both)"),
    "puzzle": (("JIGSAWPUZZLE",), "translations: game, puzzle, puzzle piece"),
    "shhh": (("SSHH",), "translations: be quiet, shh, shush (QUIET is the label quiet)"),
    "shoe": (("SHOES",), "translations: footwear, shoe, shoes, sneakers"),
    "store": (("SHOP1",), "translations: sell, shop, shopping, store"),
    "tooth": (("TEETH",), "translations: teeth, tooth"),
    "weus": (("WE", "US"), "translations: pronoun, us, we (both)"),
}

# Labels left unmapped until a reviewer decides between senses.
UNRESOLVED: dict[str, str] = {
    "can": "CAN1 means able to / possible and CAN2 has no translation; the asl-signs sense "
    "(ability or container) is unknown",
    "fall": "FALL1 is the season and FALL2 is falling down; the asl-signs sense is unknown",
    "glasswindow": "GLASS1-3 (cup, material) and WINDOW are different signs; which one "
    "asl-signs records is unknown",
}

# Why the remaining labels have no mapping (from the ASL-LEX search).
UNMAPPED_NOTES: dict[str, str] = {
    "feet": "no ASL Citizen gloss",
    "goose": "no ASL Citizen gloss",
    "pajamas": "no ASL Citizen gloss",
    "jeans": "only PANTS1/PANTS2, likely a different sign from JEANS",
    "nap": "only SLEEP, which is the label sleep",
    "pool": "only POOLGAME (billiards)",
    "puppy": "only DOG1-4, which are the label dog",
    "say": "only MENTION, a different sign",
    "sleepy": "only SLEEP (the label sleep) and TIRED, a different sign",
    "snack": "only COOKIE, DESSERT and SWEET",
    "wake": "only AWAKE, which is the label awake",
}

# Extracted but unmapped, so a reviewer can map them later without re-extracting.
REVIEW_GLOSSES: tuple[str, ...] = (
    "CAN1",
    "CAN2",
    "FALL1",
    "FALL2",
    "GLASS1",
    "GLASS2",
    "GLASS3",
    "WINDOW",
    "DRINK2",
    "EMPTY2",
    "HEAR2",
    "TALK2",
    "TOY2",
    "PANTS1",
    "PANTS2",
    "SHOP2",
    "ENTER",
    "NEED",
    "SHOULD",
    "THIS/IT",
    "GLOVES3",
    "GLOVES4",
    "TIRED",
)


def normalize(text: str) -> str:
    """Lower-case and keep only a-z and 0-9: "Thank-You" -> "thankyou"."""
    return re.sub(r"[^a-z0-9]", "", text.lower())


def variant_base(gloss: str) -> str | None:
    """The normalized word before a numeric variant suffix (CAT2 -> "cat"), or None when the
    gloss has no suffix or is a number with letters first (1DOLLAR stays as it is)."""
    match = re.fullmatch(r"([a-z][a-z0-9]*?[a-z])(\d+)", normalize(gloss))
    return match.group(1) if match else None


@dataclass(frozen=True)
class MappingRow:
    label: str
    gloss: str
    rule: str
    basis: str


@dataclass
class MappingResult:
    """Rows sorted by label then gloss; ``unmapped`` maps each label without rows to a reason."""

    rows: list[MappingRow]
    unmapped: dict[str, str]
    review: list[str]
    labels: list[str]
    by_gloss: dict[str, MappingRow] = field(default_factory=dict)

    def label_rule(self, label: str) -> str | None:
        """The strongest rule among a label's rows (exact, then variant, then synonym)."""
        rules = {row.rule for row in self.rows if row.label == label}
        return next((rule for rule in RULES if rule in rules), None)

    def coverage(self) -> dict[str, int]:
        """Label counts by strongest rule, plus mapped, unmapped and the vocabulary size."""
        counts = dict.fromkeys(RULES, 0)
        for label in self.labels:
            rule = self.label_rule(label)
            if rule is not None:
                counts[rule] += 1
        mapped = sum(counts.values())
        return {
            **counts,
            "mapped": mapped,
            "unmapped": len(self.labels) - mapped,
            "vocabulary": len(self.labels),
        }

    def candidate_glosses(self) -> list[str]:
        """Every gloss to extract: the mapped ones and the review list."""
        return sorted(set(self.by_gloss) | set(self.review))


def build_mapping(
    labels: Sequence[str], glosses: Iterable[str], *, strict: bool = True
) -> MappingResult:
    """Apply the rules to the dataset's gloss list.

    With ``strict`` (the default), a gloss named in SYNONYMS or VARIANT_EXCLUDED that the dataset
    lacks raises ValueError, so a changed dataset release is noticed rather than mapped silently.
    """
    present = sorted({gloss.strip() for gloss in glosses if gloss.strip()})
    present_set = set(present)
    if strict:
        named = {gloss for glosses_, _ in SYNONYMS.values() for gloss in glosses_}
        missing = sorted((named | set(VARIANT_EXCLUDED)) - present_set)
        if missing:
            raise ValueError(
                f"glosses named in the mapping are missing from the dataset: {missing}"
            )
    by_norm: dict[str, str] = {}
    for label in labels:
        key = normalize(label)
        if key in by_norm:
            raise ValueError(f"labels {by_norm[key]!r} and {label!r} normalize to {key!r}")
        by_norm[key] = label

    candidates: dict[str, list[MappingRow]] = {}

    def add(row: MappingRow) -> None:
        candidates.setdefault(row.gloss, []).append(row)

    for gloss in present:
        exact = by_norm.get(normalize(gloss))
        if exact is not None:
            add(MappingRow(exact, gloss, "exact", "normalized gloss equals the label"))
            continue
        base = variant_base(gloss)
        variant = by_norm.get(base) if base is not None else None
        if base is not None and variant is not None and gloss not in VARIANT_EXCLUDED:
            add(MappingRow(variant, gloss, "variant", f"numbered variant of {base.upper()}"))
    for label, (synonyms, basis) in SYNONYMS.items():
        if label not in by_norm.values():
            continue
        for gloss in synonyms:
            if gloss in present_set:
                add(MappingRow(label, gloss, "synonym", basis))

    rows: list[MappingRow] = []
    for gloss, options in sorted(candidates.items()):
        targets = sorted({row.label for row in options})
        if len(targets) > 1:
            raise ValueError(f"gloss {gloss!r} maps to more than one label: {targets}")
        if options[0].label in UNRESOLVED:
            continue
        rows.append(options[0])
    rows.sort(key=lambda row: (row.label, row.gloss))

    mapped = {row.label for row in rows}
    unmapped = {
        label: UNRESOLVED.get(label) or UNMAPPED_NOTES.get(label) or "no matching gloss"
        for label in labels
        if label not in mapped
    }
    review = sorted(set(REVIEW_GLOSSES) & present_set)
    return MappingResult(
        rows=rows,
        unmapped=unmapped,
        review=review,
        labels=list(labels),
        by_gloss={row.gloss: row for row in rows},
    )


def mapping_csv(result: MappingResult) -> str:
    """label,gloss,rule,basis,status rows (status is REVIEW_STATUS for non-exact rules)."""
    lines = ["label,gloss,rule,basis,status"]
    for row in result.rows:
        status = "automatic" if row.rule == "exact" else REVIEW_STATUS
        basis = row.basis.replace('"', "'")
        lines.append(f'{row.label},{row.gloss},{row.rule},"{basis}",{status}')
    return "\n".join(lines) + "\n"


def unmapped_text(result: MappingResult) -> str:
    """One "label: reason" line per unmapped label."""
    return "".join(f"{label}: {reason}\n" for label, reason in sorted(result.unmapped.items()))


# The 250 labels of web/public/model/meta.json, in model order.
LABELS = ["TV", "after", "airplane", "all", "alligator", "animal", "another", "any", "apple", "arm", "aunt", "awake", "backyard", "bad", "balloon", "bath", "because", "bed", "bedroom", "bee", "before", "beside", "better", "bird", "black", "blow", "blue", "boat", "book", "boy", "brother", "brown", "bug", "bye", "callonphone", "can", "car", "carrot", "cat", "cereal", "chair", "cheek", "child", "chin", "chocolate", "clean", "close", "closet", "cloud", "clown", "cow", "cowboy", "cry", "cut", "cute", "dad", "dance", "dirty", "dog", "doll", "donkey", "down", "drawer", "drink", "drop", "dry", "dryer", "duck", "ear", "elephant", "empty", "every", "eye", "face", "fall", "farm", "fast", "feet", "find", "fine", "finger", "finish", "fireman", "first", "fish", "flag", "flower", "food", "for", "frenchfries", "frog", "garbage", "gift", "giraffe", "girl", "give", "glasswindow", "go", "goose", "grandma", "grandpa", "grass", "green", "gum", "hair", "happy", "hat", "hate", "have", "haveto", "head", "hear", "helicopter", "hello", "hen", "hesheit", "hide", "high", "home", "horse", "hot", "hungry", "icecream", "if", "into", "jacket", "jeans", "jump", "kiss", "kitty", "lamp", "later", "like", "lion", "lips", "listen", "look", "loud", "mad", "make", "man", "many", "milk", "minemy", "mitten", "mom", "moon", "morning", "mouse", "mouth", "nap", "napkin", "night", "no", "noisy", "nose", "not", "now", "nuts", "old", "on", "open", "orange", "outside", "owie", "owl", "pajamas", "pen", "pencil", "penny", "person", "pig", "pizza", "please", "police", "pool", "potty", "pretend", "pretty", "puppy", "puzzle", "quiet", "radio", "rain", "read", "red", "refrigerator", "ride", "room", "sad", "same", "say", "scissors", "see", "shhh", "shirt", "shoe", "shower", "sick", "sleep", "sleepy", "smile", "snack", "snow", "stairs", "stay", "sticky", "store", "story", "stuck", "sun", "table", "talk", "taste", "thankyou", "that", "there", "think", "thirsty", "tiger", "time", "tomorrow", "tongue", "tooth", "toothbrush", "touch", "toy", "tree", "uncle", "underwear", "up", "vacuum", "wait", "wake", "water", "wet", "weus", "where", "white", "who", "why", "will", "wolf", "yellow", "yes", "yesterday", "yourself", "yucky", "zebra", "zipper"]

# ---- kaggle/asl_citizen_extract.py ----
# ASL Citizen -> MediaPipe Holistic landmarks for the edge-vision vocabulary (body of the Kaggle
# notebook kaggle/asl_citizen_extract.ipynb; scripts/build_kaggle_notebooks.py puts
# src/edgevision/data/citizen_labels.py and the 250 labels above this file).
#
# The archive (about 46 GB) is never downloaded whole: its central directory is read with HTTP
# range requests, and each video of a mapped sign is fetched on its own, decompressed in memory,
# tracked and deleted. Results go to OUT in shards of SHARD_SIZE videos; a shard is written
# through a temporary name, so a shard file on disk is complete. A rerun skips finished shards,
# also ones found under /kaggle/input (a previous version's output added as an input).
#
# Licence (Microsoft Research License Terms, use.txt in the archive): non-commercial research
# only; the data and anything derived from it (these landmarks included) must not be
# redistributed; keep the notebook and the dataset private; delete everything when the research
# is complete. See docs/DATA_LICENSES.md.

import csv
import hashlib
import io
import json
import multiprocessing
import os
import platform
import shutil
import struct
import sys
import tempfile
import time
import traceback
import urllib.error
import urllib.request
import zipfile
import zlib
from pathlib import Path

DEFAULT_URL = (
    "https://download.microsoft.com/download/b/8/8/"
    "b88c0bae-e6c1-43e1-8726-98cf5af36ca4/ASL_Citizen.zip"
)
# EV_URL points the program at a stand-in archive for local tests; leave it unset on Kaggle.
URL = os.environ.get("EV_URL") or DEFAULT_URL
# > 0: a smoke test on the first N planned videos (kaggle/asl_citizen_smoke.ipynb uses 20).
LIMIT_VIDEOS = int(os.environ.get("EV_LIMIT_VIDEOS", "0"))
SMOKE = LIMIT_VIDEOS > 0
_OUT = Path(os.environ.get("EV_OUT", "/kaggle/working/asl-citizen-landmarks"))
# A smoke test always writes to its own "-smoke" folder, so it never mixes with the real shards,
# never writes COMPLETE and is never picked up as training data (it writes SMOKE instead).
OUT = _OUT.with_name(_OUT.name + "-smoke") if SMOKE and not _OUT.name.endswith("-smoke") else _OUT
INPUT_ROOT = Path(os.environ.get("EV_INPUT", "/kaggle/input"))
SHARD_SIZE = int(os.environ.get("EV_SHARD_SIZE", "200"))
WORKERS = int(os.environ.get("EV_WORKERS", str(os.cpu_count() or 4)))
# Stop starting shards after this many hours, so a 12-hour commit still saves its output.
MAX_HOURS = float(os.environ.get("EV_MAX_HOURS", "11"))
# The smoke check: a video passes when pose and at least one hand are found in more than this
# fraction of its frames; the smoke test passes when SMOKE_PASS_FRACTION of the videos were
# tracked and that many of the tracked ones pass.
SMOKE_MOST_FRAMES = 0.5
SMOKE_PASS_FRACTION = 0.8
# "k/N": process only the shards whose index is k modulo N, so N copies of the notebook can run
# side by side; a run with every part's output as an input then merges them.
PART = os.environ.get("EV_PART", "0/1")
# Face landmarks are not used by any edge-vision model. They stay NaN in the stored layout
# unless EV_KEEP_FACE=1, which keeps the private dataset free of face meshes.
KEEP_FACE = os.environ.get("EV_KEEP_FACE", "0") == "1"
# Frames wider than this are scaled down (aspect kept) before tracking, as the live tools feed
# 640-pixel frames; landmarks are normalized to the frame, so the scale does not shift them.
MAX_WIDTH = 640
# scripts/extract_holistic.py: legacy Holistic as the asl-signs data was made (mediapipe 0.10.21,
# default detection and tracking confidence 0.5).
HOLISTIC = {
    "static_image_mode": False,
    "model_complexity": 1,
    "smooth_landmarks": True,
    "refine_face_landmarks": False,
}
# The asl-signs parquet layout: face 0-467, left_hand 468-488, pose 489-521, right_hand 522-542,
# x, y, z each, NaN where Holistic found nothing. left_hand is the signer's own left hand.
FACE, LEFT, POSE, RIGHT, N_POINTS = 0, 468, 489, 522, 543
SPLITS = ("train", "val", "test")
LAYOUT_VERSION = 1
HTTP_TRIES = 6


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


def http_range(start: int, length: int) -> bytes:
    """bytes [start, start + length) of the archive, retried with backoff."""
    request = urllib.request.Request(URL, headers={"Range": f"bytes={start}-{start + length - 1}"})
    for attempt in range(HTTP_TRIES):
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if response.status != 206:
                    raise OSError(f"HTTP {response.status} for a range request")
                data = response.read()
            if len(data) != length:
                raise OSError(f"short read: {len(data)} of {length} bytes")
            return data
        except (OSError, urllib.error.URLError) as error:
            if attempt == HTTP_TRIES - 1:
                raise
            time.sleep(2**attempt)
            log(f"retrying range {start}+{length} after {type(error).__name__}: {error}")
    raise AssertionError("unreachable")


class RemoteFile(io.RawIOBase):
    """A seekable read-only view of the archive over HTTP range requests (for zipfile)."""

    def __init__(self, size: int) -> None:
        self.size, self.position = size, 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.position

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.position, 2: self.size}[whence]
        self.position = base + offset
        return self.position

    def readinto(self, buffer) -> int:  # type: ignore[no-untyped-def]
        if self.position >= self.size:
            return 0
        length = min(len(buffer), self.size - self.position)
        data = http_range(self.position, length)
        buffer[:length] = data
        self.position += length
        return length


def archive_size() -> int:
    request = urllib.request.Request(URL, method="HEAD")
    with urllib.request.urlopen(request, timeout=60) as response:
        if response.headers.get("Accept-Ranges", "").lower() != "bytes":
            raise OSError("the server does not accept range requests; this notebook needs them")
        return int(response.headers["Content-Length"])


def member_bytes(entry: dict) -> bytes:
    """One archive member, fetched in a single range request and checked against its CRC."""
    slack = 30 + len(entry["name"].encode()) + 1024
    data = http_range(entry["offset"], slack + entry["compressed"])
    if data[:4] != b"PK\x03\x04":
        raise OSError(f"no local file header at {entry['offset']}")
    name_length, extra_length = struct.unpack("<HH", data[26:30])
    start = 30 + name_length + extra_length
    end = start + entry["compressed"]
    if end > len(data):
        data += http_range(entry["offset"] + len(data), end - len(data))
    payload = data[start:end]
    if entry["method"] == zipfile.ZIP_DEFLATED:
        payload = zlib.decompress(payload, -15)
    elif entry["method"] != zipfile.ZIP_STORED:
        raise OSError(f"unsupported compression method {entry['method']}")
    if len(payload) != entry["size"] or zlib.crc32(payload) != entry["crc"]:
        raise OSError("size or CRC mismatch after download")
    return payload


def points(landmarks, count: int):  # type: ignore[no-untyped-def]
    import numpy as np

    out = np.full((count, 3), np.nan, dtype=np.float32)
    if landmarks is not None:
        for index, point in enumerate(landmarks.landmark):
            out[index] = (point.x, point.y, point.z)
    return out


def track_video(path: str) -> dict:
    """Run Holistic over every frame; a new Holistic per video, so tracking never carries over."""
    import cv2
    import mediapipe as mp
    import numpy as np

    capture = cv2.VideoCapture(path)
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    frames = []
    width = height = 0
    holistic = mp.solutions.holistic.Holistic(**HOLISTIC)
    try:
        while True:
            ok, image = capture.read()
            if not ok:
                break
            height, width = image.shape[:2]
            if width > MAX_WIDTH:
                image = cv2.resize(
                    image,
                    (MAX_WIDTH, round(height * MAX_WIDTH / width)),
                    interpolation=cv2.INTER_AREA,
                )
            result = holistic.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            frame = np.full((N_POINTS, 3), np.nan, dtype=np.float32)
            if KEEP_FACE:
                frame[FACE:LEFT] = points(result.face_landmarks, LEFT - FACE)
            frame[LEFT:POSE] = points(result.left_hand_landmarks, POSE - LEFT)
            frame[POSE:RIGHT] = points(result.pose_landmarks, RIGHT - POSE)
            frame[RIGHT:N_POINTS] = points(result.right_hand_landmarks, N_POINTS - RIGHT)
            frames.append(frame)
    finally:
        holistic.close()
        capture.release()
    landmarks = np.stack(frames) if frames else np.zeros((0, N_POINTS, 3), dtype=np.float32)
    hands = np.isfinite(landmarks[:, LEFT, 0]) | np.isfinite(landmarks[:, RIGHT, 0])
    return {
        "landmarks": landmarks,
        "n_frames": len(frames),
        "fps": fps,
        "width": int(width),
        "height": int(height),
        "hand_frames": int(hands.sum()),
    }


def process(entry: dict) -> dict:
    """Download, track and delete one video. Errors become a status, never an exception."""
    started = time.perf_counter()
    record = {"file": entry["file"], "status": "ok", "error": ""}
    workdir = Path(tempfile.gettempdir()) / f"asl_citizen_{os.getpid()}"
    workdir.mkdir(parents=True, exist_ok=True)
    path = workdir / Path(entry["file"]).name
    try:
        try:
            payload = member_bytes(entry)
        except Exception as error:
            record.update(status="download_error", error=f"{type(error).__name__}: {error}")
            return record
        path.write_bytes(payload)
        tracked = track_video(str(path))
        record.update(tracked)
        if tracked["n_frames"] == 0:
            record.update(status="decode_error", error="no frames decoded")
    except Exception as error:
        record.update(status="track_error", error=f"{type(error).__name__}: {error}")
    finally:
        path.unlink(missing_ok=True)
        record["seconds"] = time.perf_counter() - started
    return record


def write_atomic(path: Path, data: bytes) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_bytes(data)
    temporary.replace(path)


def shard_done(directory: Path, index: int) -> bool:
    """Finished when both files exist and no video failed to download (those are retried)."""
    meta_path = directory / f"shard_{index:05d}.json"
    if not (meta_path.is_file() and (directory / f"shard_{index:05d}.npz").is_file()):
        return False
    try:
        videos = json.loads(meta_path.read_text())["videos"]
    except (OSError, ValueError, KeyError):
        return False
    return all(video["status"] != "download_error" for video in videos)


def write_shard(directory: Path, index: int, records: list[dict], plan_id: str) -> None:
    import numpy as np

    records = sorted(records, key=lambda item: item["file"])
    arrays = [r["landmarks"] for r in records if r["status"] == "ok"]
    landmarks = np.concatenate(arrays) if arrays else np.zeros((0, N_POINTS, 3), dtype=np.float32)
    starts, offset = [], 0
    for record in records:
        starts.append(offset if record["status"] == "ok" else -1)
        offset += record["n_frames"] if record["status"] == "ok" else 0
    buffer = io.BytesIO()
    np.savez_compressed(buffer, landmarks=landmarks.astype(np.float32))
    write_atomic(directory / f"shard_{index:05d}.npz", buffer.getvalue())
    videos = [
        {
            key: record.get(key)
            for key in (
                "file",
                "status",
                "error",
                "n_frames",
                "fps",
                "width",
                "height",
                "hand_frames",
                "seconds",
            )
        }
        | {"start": start}
        for record, start in zip(records, starts, strict=True)
    ]
    meta = {"plan_id": plan_id, "shard": index, "layout_version": LAYOUT_VERSION, "videos": videos}
    write_atomic(directory / f"shard_{index:05d}.json", json.dumps(meta, indent=1).encode())


def carry_over(shards: Path, plan_id: str) -> int:
    """Copy finished shards of the same plan from /kaggle/input (an earlier version's output)."""
    copied = 0
    if not INPUT_ROOT.is_dir():
        return 0
    for meta_path in sorted(INPUT_ROOT.rglob("shard_*.json")):
        target = shards / meta_path.name
        npz = meta_path.with_suffix(".npz")
        if target.exists() or not npz.is_file():
            continue
        try:
            meta = json.loads(meta_path.read_text())
        except (OSError, ValueError):
            continue
        if meta.get("plan_id") != plan_id:
            continue
        shutil.copyfile(npz, shards / npz.name)
        shutil.copyfile(meta_path, target)
        copied += 1
    return copied


def read_split_csv(archive: zipfile.ZipFile, split: str) -> list[dict]:
    text = archive.read(f"ASL_Citizen/splits/{split}.csv").decode("utf-8-sig")
    rows = list(csv.DictReader(io.StringIO(text)))
    for row in rows:
        row["split"] = split
    return rows


def write_manifest(rows: list[dict], shards: Path, mapping: "MappingResult") -> dict:  # noqa: F821
    """manifest.csv over every planned video, from the shard files."""
    status: dict[str, dict] = {}
    for meta_path in sorted(shards.glob("shard_*.json")):
        meta = json.loads(meta_path.read_text())
        for video in meta["videos"]:
            status[video["file"]] = video | {"shard": meta["shard"]}
    columns = [
        "video_file", "participant", "split", "gloss", "asl_lex_code", "label", "rule", "shard",
        "start", "n_frames", "fps", "width", "height", "hand_frames", "status", "error",
    ]  # fmt: skip
    counts: dict[str, int] = {}
    with (OUT / "manifest.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            video = status.get(row["file"], {})
            mapped = mapping.by_gloss.get(row["gloss"])
            counts[video.get("status", "missing")] = (
                counts.get(video.get("status", "missing"), 0) + 1
            )
            writer.writerow(
                {
                    "video_file": row["file"],
                    "participant": row["participant"],
                    "split": row["split"],
                    "gloss": row["gloss"],
                    "asl_lex_code": row["code"],
                    "label": mapped.label if mapped else "",
                    "rule": mapped.rule if mapped else "review",
                    "shard": video.get("shard", ""),
                    "start": video.get("start", ""),
                    "n_frames": video.get("n_frames", ""),
                    "fps": video.get("fps", ""),
                    "width": video.get("width", ""),
                    "height": video.get("height", ""),
                    "hand_frames": video.get("hand_frames", ""),
                    "status": video.get("status", "missing"),
                    "error": video.get("error", ""),
                }
            )
    return counts


def presence(landmarks) -> tuple[float, float]:  # type: ignore[no-untyped-def]
    """Fractions of frames with a pose and with at least one hand (any finite x in the block)."""
    import numpy as np

    if len(landmarks) == 0:
        return 0.0, 0.0
    pose = np.isfinite(landmarks[:, POSE:RIGHT, 0]).any(axis=1)
    hands = np.isfinite(landmarks[:, LEFT:POSE, 0]).any(axis=1) | np.isfinite(
        landmarks[:, RIGHT:N_POINTS, 0]
    ).any(axis=1)
    return float(pose.mean()), float(hands.mean())


def smoke_report(shards: Path, wall_seconds: float, total_planned: int) -> bool:
    """Per-video time and landmark presence from the written shards, and a pass/fail line."""
    import numpy as np

    rows = []
    for meta_path in sorted(shards.glob("shard_*.json")):
        meta = json.loads(meta_path.read_text())
        with np.load(meta_path.with_suffix(".npz"), allow_pickle=False) as data:
            landmarks = data["landmarks"]
        for video in meta["videos"]:
            pose = hands = 0.0
            if video["status"] == "ok":
                start, count = int(video["start"]), int(video["n_frames"])
                pose, hands = presence(landmarks[start : start + count])
            rows.append(video | {"pose": pose, "hands": hands})
    if not rows:
        log("SMOKE FAIL: no videos were processed")
        return False
    log("smoke test, per video: seconds (download + tracking, one worker), frames, fps, frames")
    log("with a pose, frames with a hand")
    for number, row in enumerate(rows, start=1):
        log(
            f"  {number:2d}. {row['status']:<14} {row.get('seconds') or 0:6.1f} s "
            f"{row.get('n_frames') or 0:4d} frames {row.get('fps') or 0:5.1f} fps  "
            f"pose {100 * row['pose']:3.0f}%  hands {100 * row['hands']:3.0f}%"
        )
    ok = [row for row in rows if row["status"] == "ok"]
    seconds = [row["seconds"] for row in rows if row.get("seconds")]
    wall_per_video = wall_seconds / len(rows)
    log(
        f"time: {np.mean(seconds):.1f} s per video per worker (median {np.median(seconds):.1f}); "
        f"wall {wall_per_video:.1f} s per video with {WORKERS} workers, so the full run of "
        f"{total_planned} videos needs about {total_planned * wall_per_video / 3600:.1f} h "
        f"({total_planned * wall_per_video / 3600 / 3:.1f} h per part with 3 parts)"
    )
    good = [
        row for row in ok if row["pose"] > SMOKE_MOST_FRAMES and row["hands"] > SMOKE_MOST_FRAMES
    ]
    if ok:
        log(
            f"landmarks: {len(ok)}/{len(rows)} videos tracked; median pose "
            f"{100 * np.median([r['pose'] for r in ok]):.0f}% and hands "
            f"{100 * np.median([r['hands'] for r in ok]):.0f}% of frames; {len(good)}/{len(ok)} "
            f"have both in more than {100 * SMOKE_MOST_FRAMES:.0f}% of frames"
        )
    tracked_enough = len(ok) >= SMOKE_PASS_FRACTION * len(rows)
    present_enough = bool(ok) and len(good) >= SMOKE_PASS_FRACTION * len(ok)
    if tracked_enough and present_enough:
        log("SMOKE PASS: downloads, tracking and shards work; run the full extraction")
        return True
    reasons = []
    if not tracked_enough:
        reasons.append(f"only {len(ok)} of {len(rows)} videos were tracked")
    if not present_enough:
        reasons.append(
            f"only {len(good)} of {len(ok)} tracked videos have a pose and a hand in most frames"
        )
    log(f"SMOKE FAIL: {'; '.join(reasons)}. Read the errors above before the full run")
    return False


README = """# asl-citizen-landmarks-private

MediaPipe Holistic landmarks for the ASL Citizen videos whose glosses map onto the 250-sign
edge-vision vocabulary, plus the glosses kept for review. Made by kaggle/asl_citizen_extract.ipynb.

PRIVATE. ASL Citizen is licensed by Microsoft for non-commercial research only. These landmarks
are derived from it and must not be shared, published or redistributed; keep this dataset and
every notebook that uses it private, and delete them when the research is complete. The licence
text is in LICENSE_ASL_Citizen.txt.

- shards/shard_NNNNN.npz: "landmarks", float32 (frames, 543, 3) for the shard's videos one after
  another, in the asl-signs parquet layout (face 0-467, left_hand 468-488, pose 489-521,
  right_hand 522-542; x, y, z; NaN where nothing was found). Face points are NaN unless the run
  set EV_KEEP_FACE=1.
- shards/shard_NNNNN.json: per video, its file, status, n_frames, fps, frame size, hand frames
  and "start", the row of its first frame in "landmarks" (-1 when it has no frames).
- manifest.csv: every planned video with participant, official split, gloss, ASL-LEX code,
  mapped label and rule (exact, variant, synonym or review), shard and status.
- label_mapping.csv, unmapped_labels.txt, coverage.json: the label mapping (non-exact rules are
  AI-assisted and unconfirmed).
- plan.json: settings and the video list per shard.
"""


def dataset_metadata() -> dict:
    owner = os.environ.get("KAGGLE_USERNAME", "<your-kaggle-username>")
    return {
        "title": "asl-citizen-landmarks-private",
        "id": f"{owner}/asl-citizen-landmarks-private",
        "licenses": [{"name": "other"}],
    }


def main() -> int:
    started = time.time()
    log(f"python {platform.python_version()} on {platform.platform()}, {os.cpu_count()} CPUs")
    import cv2
    import mediapipe as mp
    import numpy as np

    log(f"mediapipe {mp.__version__}, opencv {cv2.__version__}, numpy {np.__version__}")
    if mp.__version__ != "0.10.21":
        log("WARNING: the training data was tracked with mediapipe 0.10.21")
    if SMOKE and OUT.name.endswith("-smoke") and OUT.exists():
        shutil.rmtree(OUT)  # a smoke folder is disposable: every smoke run starts empty
    OUT.mkdir(parents=True, exist_ok=True)
    shards = OUT / "shards"
    shards.mkdir(exist_ok=True)
    free = shutil.disk_usage(OUT).free / 1e9
    log(f"output {OUT} ({free:.1f} GB free), {WORKERS} workers, shards of {SHARD_SIZE} videos")

    if SMOKE:
        log(f"SMOKE TEST: the first {LIMIT_VIDEOS} planned videos into {OUT} (not the real shards)")
    if URL != DEFAULT_URL:
        log(f"EV_URL is set: reading {URL} instead of the Microsoft Download Center (local test)")
    size = archive_size()
    log(f"ASL_Citizen.zip: {size / 1e9:.2f} GB ({size:,} bytes); range requests accepted")
    archive = zipfile.ZipFile(io.BufferedReader(RemoteFile(size), buffer_size=1 << 20))
    entries = {info.filename: info for info in archive.infolist()}
    videos = [name for name in entries if name.startswith("ASL_Citizen/videos/")]
    log(f"archive: {len(entries)} entries, {len(videos)} videos")
    (OUT / "LICENSE_ASL_Citizen.txt").write_bytes(archive.read("ASL_Citizen/use.txt"))

    rows = []
    for split in SPLITS:
        for row in read_split_csv(archive, split):
            rows.append(
                {
                    "participant": row["Participant ID"].strip(),
                    "file": row["Video file"].strip(),
                    "gloss": row["Gloss"].strip(),
                    "code": row["ASL-LEX Code"].strip(),
                    "split": split,
                }
            )
    log(f"splits: {len(rows)} videos; " + ", ".join(
        f"{s} {sum(r['split'] == s for r in rows)}" for s in SPLITS))  # fmt: skip
    mapping = build_mapping(LABELS, [row["gloss"] for row in rows])  # noqa: F821
    (OUT / "label_mapping.csv").write_text(mapping_csv(mapping))  # noqa: F821
    (OUT / "unmapped_labels.txt").write_text(unmapped_text(mapping))  # noqa: F821
    coverage = mapping.coverage()
    (OUT / "coverage.json").write_text(json.dumps(coverage, indent=1))
    log(f"label mapping: {coverage}; unmapped: {', '.join(sorted(mapping.unmapped))}")

    wanted = set(mapping.candidate_glosses())
    planned = sorted((row for row in rows if row["gloss"] in wanted), key=lambda r: r["file"])
    missing = [row["file"] for row in planned if f"ASL_Citizen/videos/{row['file']}" not in entries]
    if missing:
        log(f"WARNING: {len(missing)} planned videos are not in the archive, e.g. {missing[:3]}")
    planned = [row for row in planned if f"ASL_Citizen/videos/{row['file']}" in entries]
    total_planned = len(planned)
    if SMOKE:
        planned = planned[:LIMIT_VIDEOS]
        log(f"smoke test: {len(planned)} of the {total_planned} planned videos")
    batches = [planned[i : i + SHARD_SIZE] for i in range(0, len(planned), SHARD_SIZE)]
    settings = {
        "holistic": HOLISTIC,
        "max_width": MAX_WIDTH,
        "keep_face": KEEP_FACE,
        "layout_version": LAYOUT_VERSION,
        "shard_size": SHARD_SIZE,
        "mediapipe": mp.__version__,
    }
    plan_id = hashlib.sha256(
        json.dumps([settings, [[r["file"] for r in b] for b in batches]]).encode()
    ).hexdigest()[:16]
    plan = {
        "plan_id": plan_id,
        "settings": settings,
        "shards": [[r["file"] for r in b] for b in batches],
    }
    (OUT / "plan.json").write_text(json.dumps(plan))
    stale = [
        p
        for p in shards.glob("shard_*.json")
        if json.loads(p.read_text()).get("plan_id") != plan_id
    ]
    if stale:
        log(f"ERROR: {len(stale)} shards in {shards} come from another plan; move them away first")
        return 1
    log(f"plan {plan_id}: {len(planned)} videos in {len(batches)} shards")
    copied = 0 if SMOKE else carry_over(shards, plan_id)
    if copied:
        log(f"carried over {copied} finished shards from {INPUT_ROOT}")
    part, parts = (int(value) for value in PART.split("/"))
    if not 0 <= part < parts:
        log(f"ERROR: EV_PART={PART} must be k/N with 0 <= k < N")
        return 1
    pending = [i for i in range(len(batches)) if not shard_done(shards, i)]
    log(f"{len(batches) - len(pending)} shards done, {len(pending)} to go")
    if parts > 1:
        pending = [i for i in pending if i % parts == part]
        log(f"part {part}/{parts}: {len(pending)} of those shards are this part's")

    done_videos, video_seconds = 0, 0.0
    session_start = time.time()
    with multiprocessing.get_context("spawn" if os.name == "nt" else "fork").Pool(WORKERS) as pool:
        for number, index in enumerate(pending, start=1):
            if (time.time() - started) / 3600 > MAX_HOURS:
                log(f"stopping after {MAX_HOURS} h; rerun with this output as an input to resume")
                break
            items = []
            for row in batches[index]:
                info = entries[f"ASL_Citizen/videos/{row['file']}"]
                items.append(
                    {
                        "file": row["file"],
                        "name": info.filename,
                        "offset": info.header_offset,
                        "compressed": info.compress_size,
                        "size": info.file_size,
                        "crc": info.CRC,
                        "method": info.compress_type,
                    }
                )
            shard_started = time.time()
            records = list(pool.imap_unordered(process, items, chunksize=1))
            write_shard(shards, index, records, plan_id)
            elapsed = time.time() - shard_started
            done_videos += len(records)
            video_seconds += sum(r.get("seconds", 0.0) for r in records)
            per_1000 = (time.time() - session_start) / done_videos * 1000
            remaining = sum(len(batches[i]) for i in pending[number:])
            bad = [r for r in records if r["status"] != "ok"]
            log(
                f"shard {index + 1}/{len(batches)}: {len(records)} videos in {elapsed:.0f} s "
                f"({len(bad)} not ok), {per_1000 / 60:.1f} min per 1,000 videos, "
                f"ETA {remaining * per_1000 / 1000 / 3600:.1f} h for {remaining} videos"
            )
            for record in bad[:3]:
                log(f"  {record['file']}: {record['status']} {record['error']}")

    finished = all(shard_done(shards, i) for i in range(len(batches)))
    counts = write_manifest(planned, shards, mapping)
    (OUT / "README.md").write_text(README)
    log(f"manifest: {counts}")
    if SMOKE:
        (OUT / "COMPLETE").unlink(missing_ok=True)
        (OUT / "SMOKE").write_text(
            f"smoke test of {len(planned)} videos; not training data, never upload as a dataset\n"
        )
        smoke_report(shards, time.time() - session_start, total_planned)
        return 0
    (OUT / "dataset-metadata.json").write_text(json.dumps(dataset_metadata(), indent=1))
    if finished:
        (OUT / "COMPLETE").write_text(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
        log(f"COMPLETE: all {len(batches)} shards written to {OUT}")
    else:
        (OUT / "COMPLETE").unlink(missing_ok=True)
        own = [i for i in range(len(batches)) if i % parts == part]
        if parts > 1 and all(shard_done(shards, i) for i in own):
            log(
                f"part {part}/{parts} finished; to merge, run once more with EV_PART=0/1 and every "
                "part's output added as an input"
            )
        else:
            log("INCOMPLETE: save this version, add its output as an input and run again to resume")
    if done_videos:
        log(f"tracking: {video_seconds / done_videos:.2f} s per video per worker")
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except Exception:
        traceback.print_exc()
        sys.exit(1)
PYEOF
clean "$PY" -u "$WORK/$NAME.py" 2>&1 | tee -a "$LOG"
echo "=== extraction exit status ${PIPESTATUS[0]}" | tee -a "$LOG"
if [ -f "$OUT/COMPLETE" ]; then
  # Kaggle's own Python, where kaggle_secrets lives (not the extraction venv).
  cat > /tmp/make_private_dataset.py <<'PYEOF'
# Create, or add a version to, the PRIVATE Kaggle dataset asl-citizen-landmarks-private from a
# finished extraction (the merge run of kaggle/asl_citizen_extract.ipynb, which embeds this file;
# scripts/build_kaggle_notebooks.py puts it there). Runs with Kaggle's own Python, where
# kaggle_secrets lives:
#
#     python make_private_dataset.py /kaggle/working/asl-citizen-landmarks
#
# It reads the KAGGLE_USERNAME and KAGGLE_KEY notebook secrets, writes ~/.kaggle/kaggle.json and
# calls "kaggle datasets create -p OUT --dir-mode zip", which makes a private dataset; --public is
# never passed (the ASL Citizen licence forbids redistribution, docs/DATA_LICENSES.md). If the
# dataset exists it adds a version. When the secrets are missing or a call fails it prints the
# exact manual steps instead. It always exits 0, like the notebook cell.

import json
import os
import re
import shutil
import subprocess
import sys
from collections.abc import Callable
from pathlib import Path

SLUG = "asl-citizen-landmarks-private"
FAILURE = re.compile(r"\b(error|failed|forbidden|unauthori[sz]ed|40[0-9])\b", re.IGNORECASE)

Runner = Callable[[list[str], dict[str, str]], tuple[int, str]]


def manual_steps(out: Path, reason: str) -> str:
    return f"""
DATASET NOT CREATED AUTOMATICALLY: {reason}
Make it by hand. It must stay Private: the ASL Citizen licence forbids redistribution.
  1. Wait until this version has finished, open it and go to its Output.
  2. Choose New Dataset (or Datasets > New Dataset > Notebook Output Files) and select the
     folder {out.name}.
  3. Title: {SLUG}. Visibility: Private (the default). Create.
Or skip the dataset: in kaggle/train_v3.ipynb, Add Input > Your Work > Notebooks > this notebook
(this version). The training notebook finds manifest.csv under either kind of input.
To make the dataset automatically next time: Add-ons > Secrets, add KAGGLE_USERNAME and
KAGGLE_KEY (kaggle.com > Settings > API > Create New Token gives kaggle.json), tick both for this
notebook, then commit this merge run again.
""".strip()


def notebook_secrets() -> tuple[str, str]:
    from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]

    client = UserSecretsClient()
    return client.get_secret("KAGGLE_USERNAME"), client.get_secret("KAGGLE_KEY")


def write_credentials(user: str, key: str, home: Path) -> Path:
    directory = home / ".kaggle"
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "kaggle.json"
    path.write_text(json.dumps({"username": user, "key": key}))
    path.chmod(0o600)
    return path


def kaggle_cli() -> list[str] | None:
    found = shutil.which("kaggle")
    if not found:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=False)
        local = Path.home() / ".local" / "bin" / "kaggle"
        found = shutil.which("kaggle") or (str(local) if local.is_file() else None)
    return [found] if found else None


def run(command: list[str], env: dict[str, str]) -> tuple[int, str]:
    done = subprocess.run(command, env=env, capture_output=True, text=True, check=False)
    return done.returncode, (done.stdout + "\n" + done.stderr).strip()


def succeeded(code: int, text: str) -> bool:
    return code == 0 and not FAILURE.search(text)


def make_dataset(
    out: Path,
    *,
    secrets: Callable[[], tuple[str, str]] = notebook_secrets,
    runner: Runner = run,
    cli: Callable[[], list[str] | None] = kaggle_cli,
    home: Path | None = None,
) -> bool:
    """True when the private dataset was created or versioned; otherwise prints manual steps."""
    if not (out / "COMPLETE").is_file():
        print(manual_steps(out, f"{out} has no COMPLETE file, so the extraction is not finished"))
        return False
    if (out / "SMOKE").exists():
        print(manual_steps(out, "this is a smoke-test output, which is never a dataset"))
        return False
    try:
        user, key = secrets()
        if not user or not key:
            raise ValueError("a secret is empty")
    except Exception as error:
        reason = f"the KAGGLE_USERNAME / KAGGLE_KEY secrets are not available ({error})"
        print(manual_steps(out, reason))
        return False
    credentials = write_credentials(user, key, home or Path.home())
    print(f"wrote {credentials} (mode 600) for Kaggle user {user}")
    dataset_id = f"{user}/{SLUG}"
    metadata = {"title": SLUG, "id": dataset_id, "licenses": [{"name": "other"}]}
    (out / "dataset-metadata.json").write_text(json.dumps(metadata, indent=1))
    command = cli()
    if not command:
        print(manual_steps(out, "the kaggle command-line tool could not be installed"))
        return False
    env = dict(os.environ, KAGGLE_CONFIG_DIR=str(credentials.parent))
    # "datasets create" makes a private dataset unless --public is passed; it is never passed.
    create = [*command, "datasets", "create", "-p", str(out), "--dir-mode", "zip"]
    print("$ kaggle " + " ".join(create[len(command) :]))
    code, text = runner(create, env)
    print(text)
    if not succeeded(code, text) and re.search(r"already (in use|exists)", text, re.IGNORECASE):
        version = [*command, "datasets", "version", "-p", str(out), "--dir-mode", "zip", "-m",
                   "asl_citizen_extract merge run"]  # fmt: skip
        print(f"{dataset_id} exists; adding a version")
        code, text = runner(version, env)
        print(text)
    if not succeeded(code, text):
        print(manual_steps(out, f"the kaggle command failed (exit {code}; output above)"))
        return False
    _, status = runner([*command, "datasets", "status", dataset_id], env)
    print(f"status of {dataset_id}: {status}")
    print(
        f"private dataset: https://www.kaggle.com/datasets/{dataset_id} - processing the upload "
        "can take a few minutes; check that its page says Private"
    )
    return True


def main(argv: list[str]) -> int:
    out = Path(argv[1]) if len(argv) > 1 else Path("/kaggle/working/asl-citizen-landmarks")
    try:
        make_dataset(out)
    except Exception as error:  # the cell always exits 0; say what happened instead
        print(manual_steps(out, f"unexpected {type(error).__name__}: {error}"))
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))
PYEOF
  python /tmp/make_private_dataset.py "$OUT" 2>&1 | tee -a "$LOG"
fi
echo "=== $(date -u '+%F %T') UTC: finished; this cell always exits 0, so read $LOG" | tee -a "$LOG"
exit 0
